# HOD fitting with HODDIES

Loads the halo catalog of one run into HODDIES (dev-v2: a `Base_catalogue` of the halo array, then
a `HOD` object), as `jobs/hod_mock.sh` does, and checks it with one mock. The fit goes in the
last section.

In [ ]:
import json
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp
from halo_pipeline import require_peculiar_velocities  # halo catalogs made before the velocity fix are refused
sys.path.insert(0, str(hp.HODDIES_ROOT))
for module_name in list(sys.modules):
    if module_name == 'HODDIES' or module_name.startswith('HODDIES.'):
        del sys.modules[module_name]

import numpy as np
import matplotlib.pyplot as plt
import HODDIES
from HODDIES import HOD
from HODDIES.sim_loader import Base_catalogue
from abacus_reference_cosmology import SERIALIZABLE_PARAMETERS

print(f'Using HODDIES from {HODDIES.__file__}')

In [ ]:
from cosmoprimo.fiducial import Cosmology

abacus_cosmo = Cosmology(
    Omega0_m=SERIALIZABLE_PARAMETERS['Omega_m'],
    Omega_b=SERIALIZABLE_PARAMETERS['Omega_b'],
    h=SERIALIZABLE_PARAMETERS['h'],
    sigma8=SERIALIZABLE_PARAMETERS['sigma8'],
    n_s=SERIALIZABLE_PARAMETERS['n_s'],
    w0_fld=SERIALIZABLE_PARAMETERS['w0'],
    wa_fld=SERIALIZABLE_PARAMETERS['wa'],
    T_cmb=SERIALIZABLE_PARAMETERS['Tcmb0_K'],
    N_eff=SERIALIZABLE_PARAMETERS['Neff'],
    m_ncdm=[],
    engine='class',
)

In [ ]:
target, model, realization = 'Abacus_Small', 'LCDM', 's1234'
redshift = 0.500
param_file = hp.NOTEBOOK_DIR / f'HOD_parameter_file_z{redshift:.3f}.yaml'  # HODDIES parameter file
tracer = 'LRG'
nthreads = 32   # HODDIES threads (capped by numba's thread count); the mock of a seed depends on it
seed = 42       # HOD seed

## Halo catalog

In [ ]:
hoddies_file = hp.hoddies_catalog_file(target, model, realization, redshift)
catalog_file = hp.halo_catalog_file(target, model, realization, redshift)
if not hoddies_file.exists():
    raise FileNotFoundError(f'Missing {hoddies_file}. Run jobs/halo_finder.sh on '
                            f'{hp.run_file(target, model, realization)} at z = {redshift} first.')

halos = np.load(hoddies_file)
halo_metadata = json.loads(Path(str(catalog_file) + '.json').read_text())
require_peculiar_velocities(halo_metadata, catalog_file)
boxsize = float(halo_metadata['boxsize'])  # box size of the run, recorded by the halo finder

halo_cosmology = halo_metadata.get('cosmology') or {}
mismatches = {key: (halo_cosmology[key], expected) for key, expected in SERIALIZABLE_PARAMETERS.items()
              if key in halo_cosmology and not np.isclose(float(halo_cosmology[key]), float(expected))}
if mismatches:
    raise ValueError(f'Halo catalog cosmology does not match the AbacusSummit c000 reference: {mismatches}')

print(f'{hoddies_file}')
print(f'{len(halos):,} halos, box {boxsize} Mpc/h, log10 Mh in [{halos["log10_Mh"].min():.2f}, {halos["log10_Mh"].max():.2f}]')

## HODDIES

In [ ]:
base_catalog = Base_catalogue({name: halos[name] for name in halos.dtype.names},  # a dict: a structured array is refused
                              boxsize=boxsize, z_simu=redshift, nthreads=nthreads)
HOD_obj = HOD(base_catalog, param_file=str(param_file), nthreads=nthreads, seed=seed)
HOD_obj.cosmo = abacus_cosmo  # used for the RSD of get_wp / get_xiells
if HOD_obj.nthreads != nthreads:
    print(f'Warning: HODDIES uses {HOD_obj.nthreads} threads, not {nthreads} (numba thread count)')

print(f'{tracer} HOD parameters:')
print(json.dumps(HOD_obj.args[tracer], indent=1, default=str))

In [ ]:
cat = HOD_obj.make_mock_cat(tracers=tracer, fix_seed=seed)
n_cent = int(np.sum(cat['Central'] == 1))
print(f'{cat.size:,} galaxies ({n_cent:,} centrals, satellite fraction {1 - n_cent / cat.size:.3f}), '
      f'n = {cat.size / boxsize**3:.3e} (h/Mpc)^3')

In [ ]:
HOD_obj.HOD_plot(tracer=tracer)

## Fit

Fit settings of the parameter file (`fit_param`), used by HODDIES' own fitting tools
(`HOD_obj.initialize_fit`, `HOD_obj.run_minimizer`, scripts in `HODDIES/fit_functions/scripts/`).

In [ ]:
print(json.dumps(HOD_obj.args['fit_param'], indent=1, default=str))